In [1]:
import pandas as pd
import sqlite3
from pathlib import Path
import logging
import os
import datetime

"""
The ETL process workflow will be consist of 5 step
1. Setup, Logging & DB configuration
2. Extraction 
3. Transform 
4. Loading
5. Validation
"""

'\nThe ETL process workflow will be consist of 5 step\n1. Setup, Logging & DB configuration\n2. Extraction \n3. Transform \n4. Loading\n5. Validation\n'

In [2]:
# 1. Logging, Setup & Configuration Process

# Logging configuration
def setup_logging():
    logging.basicConfig(
        filename='etl_log.log',
        level=logging.INFO,
        format='%(asctime)s - %(levelname)s - %(message)s'
    )
    logging.info("ETL process started")


# Connect to the SQLite database using sqlite3
# Pre-defined db_path using created database
#csv_path = Path("../data")

# Creating DB connection
def connect_db(db_path):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute("PRAGMA foreign_keys = ON;")

    logging.info("Database connected")

    return conn, cursor

In [3]:
# 2. Extraction process from csv to dataframe

def extract_data(file_path):
    try:
        df_data = pd.read_csv(file_path)
        logging.info(f"Extracted {len(df_data)} rows from {file_path}")
        return df_data
    except Exception as e:
        logging.error(f"Extract failed: {e}")
        raise


In [4]:
import re

def clean_data(df_data):
    df_data = df_data.drop_duplicates()

    df_data["RegionName"] = df_data["RegionName"].fillna("OFFSHORE MALAYSIA")

    # ✅ Convert datetime
    df_data["SpudDate"] = pd.to_datetime(df_data["SpudDate"], errors="coerce")
    df_data["SubmittedAt"] = pd.to_datetime(df_data["SubmittedAt"], errors="coerce")
    df_data["WellStartDateTime"] = pd.to_datetime(df_data["WellStartDateTime"], errors="coerce")
    df_data["WellEndDateTime"] = pd.to_datetime(df_data["WellEndDateTime"], errors="coerce")
    df_data["DocumentDate"] = pd.to_datetime(df_data["DocumentDate"], errors="coerce")

    # DocumentName logic
    df_data["DocumentName"] = df_data.apply(
        lambda row: f"{row['Year']}_{row['ReportType']}_{row['WellName']}"
        if pd.isna(row["DocumentName"]) or row["DocumentName"] == ""
        else row["DocumentName"],
        axis=1
    )

    # DocumentDate fallback
    df_data["DocumentDate"] = df_data["DocumentDate"].fillna(df_data["WellEndDateTime"])

    # RigName Standardization
   
    def clean_rig_name(rig):
        if pd.isna(rig):
            return rig

        rig = rig.upper().strip()

        # Remove spaces
        rig = re.sub(r"\s+", "", rig)

        rig = re.sub(r"\s*-?\s*(\d+)$", r"-\1", rig)


        return rig
    # Clean up rig type
    def standardize_rig_type(rig_type):
        if pd.isna(rig_type):
            return None

        rig_type = rig_type.upper().strip()

        # Normalize spaces
        rig_type = re.sub(r"\s+", " ", rig_type)

        # ✅ Mapping rules
        mapping = {
            "JACK-UP": "JACK UP",
            "JACKUP": "JACK UP",
            "JACK UP": "JACK UP",

            "SEMI-SUB": "SEMI-SUBMERSIBLE",
            "SEMISUB": "SEMI-SUBMERSIBLE",
            "SEMI SUB": "SEMI-SUBMERSIBLE",

            "TENDER ASSISTED DRILLING RIG": "TENDER ASSISTED DRILLING RIG (TADR)",
            "TADR": "TENDER ASSISTED DRILLING RIG (TADR)"
        }

        return mapping.get(rig_type, rig_type) 
     
    
    df_data["RigName"] = df_data["RigName"].apply(clean_rig_name)

    # Clean up rig type
    df_data["RigType"] = df_data["RigType"].apply(standardize_rig_type)
    
    df_data = df_data[df_data["RigType"].notna()]
    df_data = df_data[df_data["RigType"] != ""]

    #RigType Standardization
    df_data["RigType"] = df_data["RigType"].str.upper().str.strip()

    #WellName Cleanup
    df_data["WellName"] = df_data["WellName"].str.replace(r"\s+", "", regex=True)

    logging.info("Data cleaned (RigName, RigType, WellName standardized)")

    return df_data

In [5]:
# 3.2 Handle null value with no information supplied, decide to keep NULL as replacing Null with value such as 0 will lead to confusion & misinterpretation. The null value are being flagged.
# 3.2 This process is to preserve data integrity

def handle_optional_metrics(df_data):
    cols = [
        "CompletionCostPlan",
        "CompletionCostActual",
        "DrillingPlanWcpf",
        "DrillingActualWcpf"
    ]

    for col in cols:
        df_data[f"{col}_is_missing"] = df_data[col].isnull().astype(int)

    logging.info("Optional metrics handled (NULL preserved + flags added)")

    return df_data


In [6]:
# 3.3 Split flat table into normalize entities & remove redundancy.

def create_dimension_tables(df_data):
    pac_df = df_data[["PacName"]].drop_duplicates()

    region_df = df_data[["RegionName"]].drop_duplicates()

    fields_df = df_data[
        ["FieldName", "PacName", "RegionName"]
    ].drop_duplicates()

    wells_df = df_data[
        [
            "WellName",
            "WellType",
            "WaterDepth",
            "SpudDate",
            "WellStartDateTime",
            "WellEndDateTime",
            "FieldName"
        ]
    ].drop_duplicates()

    rigs_df = df_data[["RigName", "RigType"]].drop_duplicates()

    logging.info("Dimension tables created (WELLS enriched with timeline data)")

    return pac_df, region_df, fields_df, wells_df, rigs_df


In [7]:
# 3.4 Seperation of transactional data from master data

def create_fact_tables(df_data):
    reports_df = df_data[
        ["ReportType", "DocumentName", "DocumentDate",
         "SubmittedAt", "SubmittedBy", "WellName"]
    ]

    drilling_df = df_data[
        ["Year", "AfeCost", "AfeDays", "FinalCost", "FinalDays",
         "WellNptPercentageWow", "WellNptPercentage",
         "CompletionCostPlan", "CompletionCostActual",
         "DrillingPlanWcpf", "DrillingActualWcpf",
         "WellName", "RigName"]
    ]

    logging.info("Fact tables created")

    return reports_df, drilling_df

In [8]:
# 4.0 Load into database process

# Load PAC
def load_pac(cursor, pac_df):
    for _, row in pac_df.iterrows():
        cursor.execute(
            "INSERT OR IGNORE INTO PAC (PACName) VALUES (?)",
            (row["PacName"],)
        )
    
    logging.info("PAC loaded")

# Load Regions
def load_regions(cursor, region_df):
    for _, row in region_df.iterrows():
        cursor.execute(
            "INSERT OR IGNORE INTO REGIONS (RegionName) VALUES (?)",
            (row["RegionName"],)
        )

    logging.info("Regions loaded")

# Load Fields
def load_fields(cursor, fields_df):
    for _, row in fields_df.iterrows():
        cursor.execute("SELECT PACId FROM PAC WHERE PACName=?", (row["PacName"],))
        pac_id = cursor.fetchone()

        cursor.execute("SELECT RegionId FROM REGIONS WHERE RegionName=?", (row["RegionName"],))
        region_id = cursor.fetchone()

        if pac_id and region_id:
            cursor.execute(
                "INSERT INTO FIELDS (FieldName, PACId, RegionId) VALUES (?,?,?)",
                (row["FieldName"], pac_id[0], region_id[0])
            )
    
    logging.info("Fields loaded")

# Load Rigs
def load_rigs(cursor, rigs_df):
    for _, row in rigs_df.iterrows():
        cursor.execute(
            "INSERT OR IGNORE INTO RIGS (RigName, RigType) VALUES (?, ?)",
            (row["RigName"], row["RigType"])
        )

    logging.info("Rigs loaded")


# Load Wells
def load_wells(cursor, wells_df):
    for _, row in wells_df.iterrows():
        cursor.execute("SELECT FieldId FROM FIELDS WHERE FieldName=?", (row["FieldName"],))
        FieldId = cursor.fetchone()

        if FieldId:
            cursor.execute("""
                INSERT INTO WELLS (
                    WellName, WellType, WaterDepth,
                    SpudDate, WellStartDateTime, WellEndDateTime,
                    FieldId
                )
                VALUES (?, ?, ?, ?, ?, ?, ?)
            """, (
                row["WellName"],
                row["WellType"],
                row["WaterDepth"],
                row["SpudDate"].strftime("%Y-%m-%d %H:%M:%S") if pd.notnull(row["SpudDate"]) else None,
                row["WellStartDateTime"].strftime("%Y-%m-%d %H:%M:%S") if pd.notnull(row["WellStartDateTime"]) else None,
                row["WellEndDateTime"].strftime("%Y-%m-%d %H:%M:%S") if pd.notnull(row["WellEndDateTime"]) else None,

                FieldId[0]
            ))
    logging.info("Wells loaded")

# Load reports
def load_reports(cursor, reports_df):
    for _, row in reports_df.iterrows():
        cursor.execute("SELECT WellId FROM WELLS WHERE WellName=?", (row["WellName"],))
        WellId = cursor.fetchone()

        if WellId:
            cursor.execute("""
                INSERT INTO REPORTS (
                    ReportType, DocumentName, DocumentDate,
                    SubmittedAt, SubmittedBy, WellId
                ) VALUES (?, ?, ?, ?, ?, ?)
            """, (
                row["ReportType"],
                row["DocumentName"],
                row["DocumentDate"].strftime("%Y-%m-%d %H:%M:%S") if pd.notnull(row["DocumentDate"]) else None,
                row["SubmittedAt"].strftime("%Y-%m-%d %H:%M:%S") if pd.notnull(row["SubmittedAt"]) else None,
                row["SubmittedBy"],
                WellId[0]
            ))
    
    logging.info("Reports loaded")

# Load Drilling Operations
def load_drilling(cursor, drilling_df):
    for _, row in drilling_df.iterrows():
        cursor.execute("SELECT WellId FROM WELLS WHERE WellName=?", (row["WellName"],))
        WellId = cursor.fetchone()

        cursor.execute("SELECT RigId FROM RIGS WHERE RigName=?", (row["RigName"],))
        RidId = cursor.fetchone()

        if WellId and RidId:
            cursor.execute("""
                INSERT INTO DRILLING_OPERATIONS (
                    Year, AfeCost, AfeDays, FinalCost, FinalDays,
                    WellNPTPercentageWOW, WellNPTPercentage,
                    CompletionCostPlan, CompletionCostActual,
                    DrillingPlanWCPF, DrillingActualWCPF,
                    WellId, RigId
                )
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            """, (
                row["Year"], row["AfeCost"], row["AfeDays"],
                row["FinalCost"], row["FinalDays"],
                row["WellNptPercentageWow"], row["WellNptPercentage"],
                row["CompletionCostPlan"], row["CompletionCostActual"],
                row["DrillingPlanWcpf"], row["DrillingActualWcpf"],
                WellId[0], RidId[0]
            ))

    logging.info("Drilling operations loaded")


In [9]:
# 4. Validation
def validate_data(df_data):
    logging.info(f"Duplicates: {df_data.duplicated().sum()}")
    logging.info(f"Null WellName: {df_data['WellName'].isnull().sum()}")

    invalid = df_data[df_data["WellEndDateTime"] < df_data["WellStartDateTime"]]
    logging.info(f"Invalid date rows: {len(invalid)}")

    print("Validation completed")

In [10]:
# 5. Main pipeline that combine all functions

def run_etl(file_path, db_name="drilling_operations.db"):
    setup_logging()
    
    conn = None  # ensure variable exists

    try:
        df = extract_data(file_path)

        df = clean_data(df)
        df = handle_optional_metrics(df)

        pac_df, region_df, fields_df, wells_df, rigs_df = create_dimension_tables(df)
        reports_df, drilling_df = create_fact_tables(df)

        conn, cursor = connect_db(db_name)

        load_pac(cursor, pac_df)
        load_regions(cursor, region_df)
        load_fields(cursor, fields_df)
        load_wells(cursor, wells_df)
        load_rigs(cursor, rigs_df)
        load_reports(cursor, reports_df)
        load_drilling(cursor, drilling_df)

        validate_data(df)

        conn.commit()

    except Exception as e:
        logging.error(f"ETL failed: {e}")
        raise

    finally:
        if conn:
            conn.close()
            logging.info("DB connection closed")


    logging.info("ETL completed successfully")
    print("ETL pipeline finished successfully")

In [11]:
run_etl("../data/DataForAssessment.csv")

C:\Users\mzulhafizal.misrat\AppData\Local\Temp\ipykernel_29440\3772985039.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_data["RegionName"] = df_data["RegionName"].fillna("OFFSHORE MALAYSIA")
C:\Users\mzulhafizal.misrat\AppData\Local\Temp\ipykernel_29440\3772985039.py:9: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  df_data["SpudDate"] = pd.to_datetime(df_data["SpudDate"], errors="coerce")
C:\Users\mzulhafizal.misrat\AppData\Local\Temp\ipykernel_29440\3772985039.py:9:

Validation completed
ETL pipeline finished successfully
